# 03 — Trend Detection
Phát hiện xu hướng và bất thường từ chuỗi thời gian thảo luận.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
from src.trend_detection.trend_analyzer import TrendAnalyzer

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Khởi tạo analyzer
analyzer = TrendAnalyzer(window=3, z_threshold=2.0)
print("✅ TrendAnalyzer đã sẵn sàng!")

## 1. Tạo sample data để demo

In [ ]:
# Tạo sample time series data với spike bất thường
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=72, freq='H')

# Tạo data với pattern và anomaly
base_value = 50
trend = np.linspace(0, 20, 72)  # Tăng nhẹ theo thời gian
seasonality = 10 * np.sin(np.linspace(0, 4*np.pi, 72))  # Seasonal pattern
noise = np.random.randn(72) * 5
values = base_value + trend + seasonality + noise

# Thêm anomaly spikes
values[20] = 150  # Spike 1
values[50] = 180  # Spike 2

# Tạo DataFrame
df_trend = pd.DataFrame({
    'timestamp': dates,
    'count': values
})

print(f"📊 Tạo sample data: {len(df_trend)} giờ")
print(f"\nThống kê:\n{df_trend['count'].describe()}")
df_trend.head()

## 2. Trực quan hóa chuỗi thời gian

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df_trend['timestamp'], df_trend['count'], linewidth=2, color='#6366f1', label='Số lượng thảo luận')
plt.scatter(df_trend['timestamp'][df_trend['count'] > 140], 
            df_trend['count'][df_trend['count'] > 140], 
            color='#ef4444', s=100, zorder=5, label='Anomaly spikes')
plt.title('Chuỗi thời gian thảo luận', fontsize=14, fontweight='bold')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("🔴 Các điểm đỏ là anomalies được phát hiện thủ công (>140)")

## 3. Phát hiện anomalies với Z-score

In [ ]:
# Detect anomalies
anomalies = analyzer.detect_anomalies(df_trend['count'])

# Thêm vào DataFrame
df_trend['is_anomaly'] = anomalies

# Visualize
plt.figure(figsize=(14, 6))
plt.plot(df_trend['timestamp'], df_trend['count'], linewidth=2, color='#6366f1', label='Normal')
plt.scatter(df_trend[df_trend['is_anomaly']]['timestamp'],
            df_trend[df_trend['is_anomaly']]['count'],
            color='#ef4444', s=100, zorder=5, label='Anomaly (Z-score)')
plt.title('Phát hiện bất thường với Z-score', fontsize=14, fontweight='bold')
plt.xlabel('Thời gian')
plt.ylabel('Số lượng')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Thống kê
n_anomalies = anomalies.sum()
print(f"\n=== PHÁT HIỆN BẤT THƯỜNG ===")
print(f"Tổng số điểm bất thường: {n_anomalies}")
print(f"Tỷ lệ: {n_anomalies / len(df_trend) * 100:.1f}%")
if n_anomalies > 0:
    print(f"\nChi tiết các điểm bất thường:")
    print(df_trend[df_trend['is_anomaly']][['timestamp', 'count']])

## 4. Tính trend score (tốc độ tăng trưởng)

In [ ]:
# Calculate trend score
trend_scores = analyzer.calculate_trend_score(df_trend['count'])
df_trend['trend_score'] = trend_scores

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: Original time series
axes[0].plot(df_trend['timestamp'], df_trend['count'], linewidth=2, color='#6366f1')
axes[0].set_title('Số lượng thảo luận', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3)

# Bottom: Trend score
axes[1].plot(df_trend['timestamp'], df_trend['trend_score'], linewidth=2, color='#22c55e')
axes[1].axhline(y=0, color='#ef4444', linestyle='--', alpha=0.5, label='Baseline')
axes[1].set_title('Trend Score (Tốc độ tăng trưởng)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Thời gian')
axes[1].set_ylabel('Growth Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Thống kê
print(f"\n=== TREND SCORE ===")
print(f"Mean: {trend_scores.mean():.2f}")
print(f"Max: {trend_scores.max():.2f}")
print(f"Min: {trend_scores.min():.2f}")
print(f"\nTop 5 thời điểm có growth rate cao nhất:")
top_growth = df_trend.nlargest(5, 'trend_score')[['timestamp', 'count', 'trend_score']]
print(top_growth.to_string(index=False))

## 5. Phân tích trending topics (multi-topic)

In [ ]:
# Tạo sample data với nhiều topics
topics_data = []
for topic in ['AI', 'Blockchain', 'Crypto', 'Electric Vehicles', 'Renewable Energy']:
    for hour in range(72):
        timestamp = datetime(2024, 1, 1) + timedelta(hours=hour)
        # Mỗi topic có pattern khác nhau
        base = np.random.randint(20, 50)
        growth = hour * (0.5 + np.random.rand() * 0.5)  # Tăng trưởng khác nhau
        count = max(0, int(base + growth + np.random.randn() * 10))
        
        # Thêm anomaly cho một số topic
        if topic == 'AI' and hour == 40:
            count = 200
        
        topics_data.append({
            'topic': topic,
            'timestamp': timestamp,
            'count': count
        })

df_topics = pd.DataFrame(topics_data)
print(f"📊 Tạo sample data: {len(df_topics)} dòng, {df_topics['topic'].nunique()} topics")
df_topics.head()

In [ ]:
# Phân tích trending topics
trending = analyzer.get_trending_topics(df_topics, top_n=5)

print("=== TOP TRENDING TOPICS ===\n")
for idx, row in trending.iterrows():
    print(f"{idx+1}. {row['topic']}")
    print(f"   Avg Growth: {row['avg_growth']:.2f}x")
    print(f"   Max Growth: {row['max_growth']:.2f}x")
    print(f"   Has Anomaly: {'🔴 Yes' if row['has_anomaly'] else '🟢 No'}")
    print(f"   Latest Count: {row['latest_count']}")
    print()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Average growth by topic
axes[0].barh(trending['topic'][::-1], trending['avg_growth'][::-1], color='#6366f1')
axes[0].set_title('Trung bình tốc độ tăng trưởng', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Avg Growth Rate')
for i, v in enumerate(trending['avg_growth'][::-1]):
    axes[0].text(v + 0.05, i, f'{v:.2f}x', va='center', fontweight='bold')

# Right: Latest count by topic
axes[1].barh(trending['topic'][::-1], trending['latest_count'][::-1], color='#22d3ee')
axes[1].set_title('Số lượng thảo luận mới nhất', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Latest Count')
for i, v in enumerate(trending['latest_count'][::-1]):
    axes[1].text(v + 2, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. So sánh với dữ liệu thực tế (nếu có)

In [ ]:
# Load dữ liệu thực tế nếu có
DATA_PATH = Path('../data/processed/youtube_comments_sentiment.csv')

if DATA_PATH.exists():
    print(f"📊 Đang đọc dữ liệu từ {DATA_PATH}...")
    df_real = pd.read_csv(DATA_PATH)
       
    # Tìm cột thời gian
    time_col = None
    for col in ['published_at', 'published_time', 'crawled_at']:
        if col in df_real.columns:
            time_col = col
            break
    
    if time_col:
        df_real[time_col] = pd.to_datetime(df_real[time_col], errors='coerce')
        df_real = df_real.dropna(subset=[time_col])
        df_real['hour'] = df_real[time_col].dt.floor('H')
        
        # Group by hour
        hourly_counts = df_real.groupby('hour').size().reset_index(name='count')
        
        print(f"✅ Đã xử lý {len(hourly_counts)} giờ có dữ liệu")
        
        # Detect anomalies
        if len(hourly_counts) > 10:
            anomalies = analyzer.detect_anomalies(hourly_counts['count'])
            n_anomalies = anomalies.sum()
            
            print(f"\n🔍 Phát hiện {n_anomalies} điểm bất thường")
            
            # Visualize
            plt.figure(figsize=(14, 6))
            plt.plot(hourly_counts['hour'], hourly_counts['count'], linewidth=2, color='#6366f1')
            if n_anomalies > 0:
                plt.scatter(hourly_counts[anomalies]['hour'],
                           hourly_counts[anomalies]['count'],
                           color='#ef4444', s=100, zorder=5, label='Anomaly')
            plt.title('Thảo luận theo thời gian (dữ liệu thực tế)', fontsize=14, fontweight='bold')
            plt.xlabel('Thời gian')
            plt.ylabel('Số bình luận')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        else:
            print("⚠️  Không đủ dữ liệu để phân tích trend")
else:
    print(f"⚠️  Không tìm thấy file: {DATA_PATH}")
    print("Hãy chạy crawler và sentiment analyzer trước!")
    print("\n📌 Sử dụng sample data ở trên để demo trend detection.")

## 7. Kết luận

In [ ]:
print("=== KẾT LUẬN TREND DETECTION ===\n")

print("✅ Trend Detection sử dụng:")
print("   - Z-score để phát hiện bất thường")
print("   - Rolling window để tính growth rate")
print("   - Rule-based để xác định trending topics")
    
print("\n📊 Ứng dụng:")
print("   - Cảnh báo sớm khi có spike bất thường")
print("   - Xác định chủ đề đang thịnh hành")
print("   - Theo dõi xu hướng theo thời gian thực")
    
print("\n⚠️  Hạn chế:")
print("   - Phụ thuộc vào ngưỡng Z-score (cần tune)")
print("   - Chỉ phát hiện được spike đột biến")
print("   - Chưa có seasonal decomposition")
    
print("\n🔜 Cải tiến trong tương lai:")
print("   - ARIMA/SARIMA cho time series forecasting")
print("   - Prophet (Facebook) cho trend detection")
print("   - LSTM/Transformer cho prediction")
    
print("\n✅ Trend detection hoàn thành!")